In [ ]:
!pip -q install -U pip setuptools wheel
!pip -q install "transformers==4.56.2" "sentence-transformers==3.0.1" "huggingface-hub<1.0" "tokenizers==0.22.2"
!pip -q install langchain langchain-groq pypdf
!pip -q install llama-index llama-index-llms-groq llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
import getpass
import os

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

Enter API key for Groq: ··········


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# our data folder (PDF/TXT/MD in this folder)
DATA_DIR = "/content/drive/MyDrive/data"

print("DATA_DIR:", DATA_DIR)

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/data


In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq as LI_Groq

In [ ]:
# LlamaIndex global
Settings.llm = LI_Groq(model="llama-3.3-70b-versatile", temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.node_parser = SentenceSplitter(chunk_size=600, chunk_overlap=100)

In [ ]:
# Load docs from Google Drive folder
try:
    docs = SimpleDirectoryReader(input_dir=DATA_DIR).load_data()
except Exception as e:
    docs = []
    print("[WARN] Could not load docs:", e)

print("Docs loaded:", len(docs))
if len(docs) == 0:
    print(f"[WARN] Put PDF/TXT/MD files in: {DATA_DIR}")

# Build in-memory vector index
index = VectorStoreIndex.from_documents(docs) if docs else None

# Retriever (raw chunks)
retriever = index.as_retriever(similarity_top_k=4) if index else None

Docs loaded: 156


In [ ]:
from typing import List
# Extracts page number safely from varying metadata formats.
def _page_from_meta(meta: dict) -> str:
    # LlamaIndex metadata keys vary by reader; try common ones
    for k in ["page_label", "page_number", "page", "page_idx"]:
        if k in meta and meta[k] is not None:
            return str(meta[k])
    return "?"

def retrieve_with_citations(query: str, top_k: int = 5, max_chars: int = 650) -> str:
    """
    Returns chunks with citations like: [SOURCE: file.pdf p.3 | score=0.812] ...
    """
    if retriever is None:
        return "No index available. Add docs to DATA_DIR and rebuild index."
    # temporarily override top_k if desired
    hits = retriever.retrieve(query)[:top_k]
    out: List[str] = []
    for h in hits:
        meta = h.node.metadata or {}
        src = meta.get("file_name", "doc")
        page = _page_from_meta(meta)
        txt = (h.node.get_content() or "").strip().replace("\n", " ")
        out.append(f"[SOURCE: {src} p.{page} | score={h.score:.3f}] {txt[:max_chars]}")
    return "\n\n".join(out) if out else "No relevant chunks found."

##PART A — Vanilla RAG (single retrieval → single answer with citations)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

def vanilla_rag(question: str) -> str:

    retrieval_query = (
        f"{question}\n"
        f"Focus on survey paper sections like: taxonomy, challenges, limitations, future work, open problems."
    )

    # ---- RETRIEVE FIRST ----
    context = retrieve_with_citations(retrieval_query, top_k=5)

    # ---- PRINT RAW CHUNKS BEFORE LLM ----
    print("\n" + "="*80)
    print("RETRIEVED CHUNKS (Before LLM)")
    print("="*80)
    print(context)
    print("="*80 + "\n")

    # ---- NOW PASS TO LLM ----
    prompt = f"""
You are VANILLA RAG.
Use ONLY the CONTEXT. If the answer is not supported by the context, say:
"Not found in documents."

Return:
1) Answer in 4–6 sentences.
2) Citations: list the SOURCE lines (file + page) you used.

QUESTION:
{question}

CONTEXT:
{context}
"""

    return llm.invoke(prompt).content


# ---- Example ----
qA = "According to these surveys, what are the most common challenges in evaluating agentic RAG systems?"
print(vanilla_rag(qA))


RETRIEVED CHUNKS (Before LLM)
[SOURCE: Agentic_AI_A_Comprehensive_Survey_of_Technologies_Applications_and_Societal_Implications.pdf p.3 | score=0.454] Ashis: Agentic AI: A Comprehensive Survey of Technologies, Applications, and Societal Implications TABLE 3. Comparison table of the proposed work with existing works Author(s) & Y ear Agentic AI Comparison Metho- dologies Type of AI Agents Working of Agents Techno- logies Used Appli- cations Impact on Labour and Society Challenges with agentic AI Open Prob- lems Remarks Traditional AI AI Agents Olujimi et al. [90] 2025 BC ND BC BC ND BC BC ND ND BC BC Potential impact on SMMEs Jisna et al. [93] 2025 BC ND ND BC BC BC BC BC ND BC ND Working Principles, advantages Kamalov et al. [91] 2025 BC ND ND BC ND ND BC BC ND BC ID Agentic AI in education

[SOURCE: 2508.17692v1.pdf p.23 | score=0.445] LLM-based Agentic Reasoning Frameworks: A Survey from Methods to Scenarios 111:23 refinement. Table 4 summarize different metrics and datasets used in

## PART B — Tool-Augmented Agent (retrieval tool + calculator) with citations

In [ ]:
import math
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

@tool
def calculator(expr: str) -> str:
    """Evaluate simple math: '12*7', 'sqrt(81)'."""
    try:
        return str(eval(expr, {"__builtins__": {}}, {"sqrt": math.sqrt, "pow": pow}))
    except Exception as e:
        return f"Calc error: {e}"

@tool
def private_docs_retriever(query: str) -> str:
    """
    Search private survey papers and return relevant chunks with citations (file + page).
    Use for: definitions, taxonomy, limitations, open problems, comparisons.
    """
    return retrieve_with_citations(query, top_k=5)

tools = [calculator, private_docs_retriever]

tool_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a TOOL-AUGMENTED assistant for survey papers.\n"
        "Rules (must follow):\n"
        "1) If the user asks any arithmetic, you MUST call calculator and output the numeric result.\n"
        "2) If the user asks for claims from documents, you MUST call private_docs_retriever.\n"
        "3) Final output MUST have exactly 2 sections in this order:\n"
        "   A) Math: <number>\n"
        "   B) Open problems (3 lines):\n"
        "      - <open problem> (Cite: file p.X)\n"
        "      - <open problem> (Cite: file p.X)\n"
        "      - <open problem> (Cite: file p.X)\n"
        "If evidence is missing, write: 'Not found in documents' (no invention).\n"
        "Never invent citations."
    ),
)


qB = "Math: 12*3. Docs: list 3 key open problems in agentic AI from my survey papers with citations."
res = tool_agent.invoke({"messages": [{"role": "user", "content": qB}]})
print(res["messages"][-1].content)

Math: 36
Open problems (3 lines):
- Transparent agent (Cite: 2508.11126v1.pdf p.24)
- Agentic AI poses challenges like scalability issues, human-AI misalignment, energy inefficiency, ethical risks, poor interpretability, and security concerns (Cite: Agentic_AI_A_Comprehensive_Survey_of_Technologies_Applications_and_Societal_Implications.pdf p.11)
- Lack of causal reasoning, inherited LLM constraints (e.g., hallucinations, shallow reasoning), incomplete agentic properties (e.g., autonomy, proactivity), and failures in long-horizon planning and recovery (Cite: 2505.10468v4.pdf p.21)


## PART C: Agentic RAG (Planning + Sub-queries + Verify + Refine)


In [ ]:
from typing import List, Dict, Any
import re

from langchain_core.tools import tool
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# -----------------------------
# TOOL: Planner (Sub-queries)
# -----------------------------
@tool
def rag_planner(question: str) -> str:
    """
    Create 2–4 focused retrieval sub-queries for survey papers.
    Output one query per line, no numbering.
    """
    prompt = f"""
You are a RAG query planner for survey research papers on agentic AI / agentic RAG.

Create 2 to 4 focused retrieval queries to answer the question.

Rules:
- Each query should be short (6–14 words).
- Include survey cues when helpful: taxonomy, evaluation, benchmarks, limitations, future work, open problems.
- Output ONLY the queries, one per line, no numbering.

QUESTION:
{question}
"""
    return llm.invoke(prompt).content.strip()


# -----------------------------
# TOOL: Evidence Critic
# -----------------------------
@tool
def evidence_critic(payload: str) -> str:
    """
    Decide if evidence sufficiently answers the question.
    If insufficient, propose a better retrieval query.
    Output EXACTLY:
      DECISION: SUFFICIENT or INSUFFICIENT
      MISSING: <what is missing>
      BETTER_QUERY: <one improved query>
    """
    prompt = f"""
You are an Evidence Critic for survey-paper Agentic RAG.

Output EXACTLY:
DECISION: SUFFICIENT or INSUFFICIENT
MISSING: <what is missing in evidence>
BETTER_QUERY: <one improved retrieval query that targets survey sections/keywords>

Criteria:
- SUFFICIENT only if evidence directly supports the requested claims.
- INSUFFICIENT if evidence is generic/off-topic or lacks survey cues (taxonomy/challenges/limitations/future work/evaluation).

Hint terms to use in BETTER_QUERY when needed:
taxonomy, benchmarks, evaluation, limitations, future work, open problems, agentic RAG, planning, verification, grounding, hallucination.

PAYLOAD:
{payload}
"""
    return llm.invoke(prompt).content.strip()


# -----------------------------
# Helper: parse planner output
# -----------------------------
def _parse_lines(text: str) -> List[str]:
    lines = []
    for line in text.splitlines():
        s = line.strip()
        if not s:
            continue
        s = re.sub(r"^\s*[\-\*\d\.\)]\s*", "", s)  # remove bullets/numbering
        if s:
            lines.append(s)
    # keep 2–4
    return lines[:4] if len(lines) > 4 else lines


# -----------------------------
# Helper: parse critic output
# -----------------------------
def _parse_critic(text: str) -> Dict[str, str]:
    out = {"decision": "", "missing": "", "better_query": ""}
    for line in text.splitlines():
        if line.startswith("DECISION:"):
            out["decision"] = line.split(":", 1)[1].strip().upper()
        elif line.startswith("MISSING:"):
            out["missing"] = line.split(":", 1)[1].strip()
        elif line.startswith("BETTER_QUERY:"):
            out["better_query"] = line.split(":", 1)[1].strip()
    return out


# -----------------------------
# Final synthesis (grounded)
# -----------------------------
def _final_answer(question: str, evidence: str) -> str:
    prompt = f"""
You must answer using ONLY the EVIDENCE.

Task:
- Provide exactly 5 bullet points comparing vanilla RAG vs agentic RAG.
- Each bullet MUST end with: (Cite: <file> p.<page>)
- Then include:
  Evidence:
  - 2–4 short quotes (each with Cite)
  Citations:
  - unique (file p.page)

If evidence does not support the comparison, say:
Not found in documents
and briefly state what is missing.

QUESTION:
{question}

EVIDENCE:
{evidence}
"""
    return llm.invoke(prompt).content.strip()


# ============================================================
# MAIN: Visible Agentic RAG Controller
# ============================================================
def agentic_rag_visible(question: str, max_refine_rounds: int = 1, top_k_per_query: int = 5) -> str:
    """
    Prints:
      PLAN → sub-queries → RETRIEVE (each) → VERIFY → REFINE (optional) → FINAL
    """
    print("\n" + "=" * 90)
    print("QUESTION:", question)
    print("=" * 90)

    # ---------- PLAN ----------
    print("\n PLANNING (creating sub-queries)...")
    plan_text = rag_planner.invoke(question)  # tool call
    sub_queries = _parse_lines(plan_text)

    if not sub_queries:
        sub_queries = [question]

    print(" SUB-QUERIES:")
    for i, sq in enumerate(sub_queries, 1):
        print(f"  {i}. {sq}")

    # ---------- RETRIEVE (multi) ----------
    evidence_blocks: List[str] = []
    for i, sq in enumerate(sub_queries, 1):
        print(f"\n RETRIEVE {i}/{len(sub_queries)}:", sq)
        ev = private_docs_retriever.invoke(sq)  # tool call (must return file/page citations)
        print(ev[:900] + (" ..." if len(ev) > 900 else ""))
        evidence_blocks.append(ev)

    evidence_all = "\n\n".join(evidence_blocks)

    # ---------- VERIFY ----------
    print("\n VERIFY (is evidence sufficient?)")
    critic_payload = f"QUESTION:\n{question}\n\nEVIDENCE:\n{evidence_all}"
    critic_text = evidence_critic.invoke(critic_payload)  # tool call
    print(critic_text)

    crit = _parse_critic(critic_text)

    # ---------- REFINE (optional, once by default) ----------
    rounds = 0
    while crit.get("decision") != "SUFFICIENT" and rounds < max_refine_rounds:
        rounds += 1
        bq = crit.get("better_query") or ""
        if not bq:
            break

        print("\n REFINE ROUND", rounds)
        print(" BETTER_QUERY:", bq)

        refined = private_docs_retriever.invoke(bq)  # tool call
        print(refined[:900] + (" ..." if len(refined) > 900 else ""))

        evidence_all = evidence_all + "\n\n" + refined

        print("\n RE-VERIFY")
        critic_payload = f"QUESTION:\n{question}\n\nEVIDENCE:\n{evidence_all}"
        critic_text = evidence_critic.invoke(critic_payload)
        print(critic_text)
        crit = _parse_critic(critic_text)

    # ---------- FINAL ANSWER ----------
    print("\nFINAL ANSWER (grounded synthesis)\n")
    answer = _final_answer(question, evidence_all)
    print(answer)

    # If still insufficient, show what was missing
    if crit.get("decision") != "SUFFICIENT":
        missing = crit.get("missing", "").strip()
        if missing:
            print("\n STILL INSUFFICIENT — MISSING:\n", missing)

    return answer


# ============================================================
# RUN (your original question)
# ============================================================
qC = "Based on my survey papers, compare vanilla RAG vs agentic RAG in 5 bullet points, each with a citation."
agentic_rag_visible(qC, max_refine_rounds=1)


QUESTION: Based on my survey papers, compare vanilla RAG vs agentic RAG in 5 bullet points, each with a citation.

 PLANNING (creating sub-queries)...
 SUB-QUERIES:
  1. What are the key differences between vanilla RAG and agentic RAG models?
  2. What are the evaluation metrics used to compare vanilla RAG and agentic RAG performance?
  3. How do vanilla RAG and agentic RAG models perform on benchmark datasets for agentic tasks?
  4. What are the limitations of using vanilla RAG versus agentic RAG in agentic AI applications?

 RETRIEVE 1/4: What are the key differences between vanilla RAG and agentic RAG models?
[SOURCE: 2505.10468v4.pdf p.26 | score=0.342] This retrieval- based grounding mechanism is particularly effective in domains such as enterprise search and customer support, where accuracy and access to up-to-date knowledge are essential for reliable task execution and user trust. In Agentic AI systems, RAG serves as a shared ground- ing mechanism across agents. For example, a 

'Based on the provided evidence, here are 5 bullet points comparing vanilla RAG vs agentic RAG:\n\n• Agentic RAG is a shared grounding mechanism across agents, enabling them to retrieve external knowledge dynamically and enhance the relevance and context of their outputs (Cite: 2505.10468v4.pdf p.26)\n• Agentic RAG allows distributed agents to operate on a unified semantic layer, avoiding or minimizing inconsistencies due to divergent contextual views (Cite: 2505.10468v4.pdf p.26)\n• Agentic RAG is particularly significant in conversational agents and real-time decision-making systems (Cite: Agentic_AI_Autonomous_Intelligence_for_Complex_GoalsA_Comprehensive_Survey.pdf p.18917)\n• Agentic RAG empowers agents to retrieve external knowledge dynamically, enhancing the relevance and context of their outputs (Cite: Agentic_AI_Autonomous_Intelligence_for_Complex_GoalsA_Comprehensive_Survey.pdf p.18917)\n• Agentic RAG is a key component of agentic AI systems, enabling them to retain contextua